In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

EPOCHS = 50
BATCH = 16
IMG_SIZE = 224
NUM_CLASSES = 38


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

DEVICE: cpu


In [12]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
])

test_ds  = datasets.ImageFolder(r"D:\praca_dyplomowa_magisterska_repo\praca_dyplomowa_magisterskie\old\michal\nowe\100\test", transform=test_val_transform)

with open("classes.txt", "w", encoding="utf-8") as f:
        for class_name in test_ds.classes:
                f.write(class_name + "\n")

test_loader  = DataLoader(test_ds, batch_size=BATCH)


In [13]:

import os

file_name =r"D:\praca_dyplomowa_magisterska_repo\praca_dyplomowa_magisterskie\output\mobilenetv3_100.pth"


model = models.mobilenet_v3_small(weights=None)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 38)
state_dict = torch.load(file_name, map_location=device)
if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]

model.load_state_dict(state_dict)
model.to(device)
model.eval()

dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    "mobilenetv3msall.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    dynamo=False
)

C:\Users\kubac\AppData\Local\Temp\ipykernel_10120\506801843.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [1]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(r"D:/praca_dyplomowa_magisterska_repo/praca_dyplomowa_magisterskie\flutter\flutter_application_1/saved_model")

tflite_model = converter.convert()

with open("mobilenetv3small.tflite", "wb") as f:
    f.write(tflite_model)